# 第 8 章 · 真实模型接入：LitellmModel

**这一章你会得到什么**：看清一个真实的 Model 实现（`LitellmModel`）在 `query()` 里到底做了哪几件事——调 API、算成本、解析动作、组装带 `extra` 的消息。**本章不联网、不花钱**：我们只跑不需要网络的部分，API 调用用源码阅读代替。

## 📖 对照源码（在 IDE 里打开这些文件，边看边跑）

- `src/minisweagent/models/litellm_model.py` **L81–105** — `query()`（retry→调API→算成本→解析→组装消息）
- `src/minisweagent/models/litellm_model.py` **L107–125** — `_calculate_cost`（成本追踪）
- `src/minisweagent/models/litellm_model.py` **L127–134** — `_parse_actions`（离线可测的那步）
- `src/minisweagent/models/__init__.py` **L45–62 / L92–113** — `get_model` / `get_model_class`（模型工厂）

> 快捷：代码格里 `函数名??` 直接打印源码；或用 `show_source("相对路径", 起始行, 结束行)`。

In [ ]:
import os, sys
from pathlib import Path
os.environ["MSWEA_SILENT_STARTUP"] = "1"
REPO = Path(r"/Users/xinranzhao/Documents/llm-study/books/mini-swe-agent-source-guide/mini-swe-agent")
SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
os.chdir(REPO)
import minisweagent
print("mini-SWE-agent:", minisweagent.__version__)

In [ ]:
def show_source(rel_path: str, start: int, end: int) -> None:
    lines = (REPO / rel_path).read_text().splitlines()
    end = min(end, len(lines))
    w = len(str(end))
    for n in range(start, end + 1):
        print(f"{n:>{w}}  {lines[n - 1]}")

## 概念：`query()` 的五个动作

1. `retry(...)` 包裹 `_query()`（带指数退避重试）
2. `_query()` 调 `litellm.completion(model, messages, tools=[BASH_TOOL], ...)`
3. `_calculate_cost()` 算这次调用的成本，累加进 `GLOBAL_MODEL_STATS`
4. `_parse_actions()` 从响应里解析 tool calls（失败抛 `FormatError`，且**必须**把原始 response 挂到异常上）
5. 组装 assistant 消息，`extra` 里塞 actions / response / cost / timestamp

In [ ]:
show_source("src/minisweagent/models/litellm_model.py", 81, 105)

## 实验 1：不联网，创建一个 LitellmModel 并看它的配置

`__init__` 不发任何网络请求，所以可以安全创建。看它默认的 observation / format_error 模板从哪来。

In [ ]:
from minisweagent.models.litellm_model import LitellmModel
m = LitellmModel(model_name="gpt-4o", cost_tracking="ignore_errors")
print("model_name:", m.config.model_name)
print("observation_template 前 60 字:", repr(m.config.observation_template[:60]))
tv = m.get_template_vars()
print("get_template_vars keys:", list(tv)[:6], "...")

## 实验 2：`format_message` —— 统一的消息构造

Agent 的 `run()` 就是靠它构造 system / user 消息。空 multimodal_regex 时它基本是透传。

In [ ]:
print(m.format_message(role="system", content="你是助手"))
print(m.format_message(role="user", content="任务：修 bug"))

## 实验 3：`_parse_actions` —— 不联网也能测解析

真实 API 的响应对象有 `choices[0].message.tool_calls` 和 `choices[0].finish_reason`。
我们用 `SimpleNamespace` 伪造一个响应，直接喂给 `_parse_actions`——这正是网络那步之后发生的事。

In [ ]:
from types import SimpleNamespace
tc = SimpleNamespace(id="call_1", function=SimpleNamespace(name="bash", arguments='{"command": "pytest -q"}'))
fake_response = SimpleNamespace(
    choices=[SimpleNamespace(message=SimpleNamespace(tool_calls=[tc]), finish_reason="tool_calls")]
)
print(m._parse_actions(fake_response))

## 动手：伪造一个“没有 tool call”的响应

补全 `bad_response`（`tool_calls=None`，`finish_reason="stop"`），喂给 `_parse_actions`，
观察它抛出的 `FormatError`。想一下：为什么 harness 宁可报错也不放行？

In [ ]:
from minisweagent.exceptions import FormatError
# TODO: 造一个 choices[0].message.tool_calls = None 的响应
bad_response = ...
# 写完后取消注释：
# try:
#     m._parse_actions(bad_response)
# except FormatError as e:
#     print("FormatError:", e.messages[0]["content"])

## 观察点
- **成本追踪是一等公民**：`_calculate_cost` 失败默认会 **raise**（除非 `cost_tracking="ignore_errors"`），因为跑评测时“算错钱”比“报错”更危险。
- `_parse_actions` 失败时源码特意把 `response` 挂到异常的 `extra` 上——保证轨迹里能复原“模型当时到底吐了什么”。这是可观测性的硬约束。
- `LitellmModel` 和第 5 章的 `DeterministicToolcallModel` 提供**完全相同的方法**（`query/format_message/format_observation_messages/...`），所以能互换——这就是 Protocol 的回报。

## 闭卷检查
1. `query()` 的五个动作分别是什么？
2. 为什么解析失败要把 response 持久化到异常上？
3. 成本算不出来时，默认行为是什么？为什么这样设计？